# Smart MCQ Solver 
## Applying knowledge from Milestone 3 and 4

**Pipeline overview:**
1. **RAG retrieval:**
   - FAISS bi-encoder (all-MiniLM-L6-v2)
   - top-5 candidates
   - cross-encoder rerank (ms-marco-MiniLM-L-6-v2)
   - best context document
2. **RAG-augmented formatting:**
   - `Context: [doc] [SEP] prompt [SEP] option` for each of the 5 options
3. **LoRA fine-tuned BERT:**
   - AutoModelForMultipleChoice fine-tuned on train.csv with the RAG-augmented inputs
4. **Length tiebreaker:**
   -  content-word length signal blended in when model probabilities are close
5. **MAP@3 submission:**
    - top-3 options by blended score to `submission.csv`

Scores so far:
- V1 (dummy): 0.16 | V2 (TF-IDF cosine): 0.29 | V3 (content-word length): 0.48 | V4 (length+cosine blend): 0.495 | V5 (length+MiniLM): 0.483

# 0. Environment setup - run first

In [1]:
# Upgrade packages that have version conflicts on Kaggle
!pip install -q torchao --upgrade
!pip install -q peft --upgrade
!pip install -q faiss-cpu
!pip install -q wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 34.3 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 12.7 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 99.6 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 r

In [2]:
# Dependencies and Libraries
import os, string, numpy as np, pandas as pd, torch
import torch.nn.functional as F
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForMultipleChoice, TrainingArguments, Trainer
from datasets import Dataset
from peft import get_peft_model, LoraConfig, TaskType
import faiss

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")

Skipping import of cpp extensions due to incompatible torch version. Please upgrade to torch >= 2.11.0 (found 2.10.0+cu128).


/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv
Device: cuda


# 1. Load data

In [3]:
DATA_DIR   = "/kaggle/input/competitions/smart-mcq-solver-challenge"

train      = pd.read_csv(f"{DATA_DIR}/train.csv")
test       = pd.read_csv(f"{DATA_DIR}/test.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

OPTION_COLS = ['A', 'B', 'C', 'D', 'E']
LABEL_MAP   = {'A': 0, 'B': 1, 'C': 2, 'D': 3, 'E': 4}
IDX_TO_OPT  = {v: k for k, v in LABEL_MAP.items()}

train['label'] = train['answer'].map(LABEL_MAP)

print(f"train: {train.shape}  test: {test.shape}")
train.head(3)

train: (2000, 9)  test: (500, 7)


,id,prompt,A,B,C,D,E,answer,label
0,1,Pick the best possible answer: What is Martin ...,Martin Heidegger believes that humans exist wi...,Martin Heidegger believes that humans do not e...,Martin Heidegger does not believe in the exist...,Martin Heidegger believes that the relationshi...,Martin Heidegger believes that time is an illu...,B,1
1,2,What is accelerator-based light-ion fusion?,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,Accelerator-based light-ion fusion is a techni...,A,0
2,3,Determine the correct option: What is the term...,Blueshifting,Redshifting,Reddening,Whitening,Yellowing,C,2


## Quick sanity checks

In [4]:
train.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
answer    0
label     0
dtype: int64

In [5]:
test.isna().sum()

id        0
prompt    0
A         0
B         0
C         0
D         0
E         0
dtype: int64

# 2. Build the RAG knowledge base + FAISS index

The KB is built from train.csv correct answers only (same setup as Milestone 3).
The bi-encoder (all-MiniLM-L6-v2) embeds all KB documents once, up front.

In [6]:
print("Building knowledge base from correct answers...")
kb = [str(row[row['answer']]) for _, row in train.iterrows()]
print(f"KB size: {len(kb)} documents")

print("Loading bi-encoder and building FAISS index...")
bi_encoder = SentenceTransformer('all-MiniLM-L6-v2', device=DEVICE)
kb_embeddings = bi_encoder.encode(kb, batch_size=64,
                                  show_progress_bar=True,
                                  normalize_embeddings=True)
kb_embeddings = kb_embeddings.astype(np.float32)

# Inner-product index (equivalent to cosine similarity on L2-normalized vectors)
faiss_index = faiss.IndexFlatIP(kb_embeddings.shape[1])
faiss_index.add(kb_embeddings)

print(f"FAISS index ready — {faiss_index.ntotal} vectors")

Building knowledge base from correct answers...
KB size: 2000 documents
Loading bi-encoder and building FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

FAISS index ready — 2000 vectors


In [7]:
print("Loading cross-encoder for reranking...")
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2', device=DEVICE)
print("Cross-encoder ready.")

Loading cross-encoder for reranking...


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

Cross-encoder ready.


# 3. RAG retrieval function

For each prompt: bi-encoder → FAISS top-5 → cross-encoder rerank → return the single best document.

In [8]:
def retrieve_best_doc(prompts, k=5):
    """
    Given a list of prompt strings, return the single best KB document
    for each prompt via two-stage retrieval:
      1. FAISS bi-encoder → top-k candidates
      2. Cross-encoder rerank → pick highest-scoring doc

    Returns a list of strings, one best-doc per prompt.
    """
    # Encode all prompts at once
    prompt_embs = bi_encoder.encode(prompts, batch_size=64,
                                    show_progress_bar=False,
                                    normalize_embeddings=True).astype(np.float32)

    _, indices = faiss_index.search(prompt_embs, k)  # (n, k)

    best_docs = []
    for i, (prompt, cand_indices) in enumerate(zip(prompts, indices)):
        docs_k = [kb[j] for j in cand_indices]
        pairs  = [[prompt, doc] for doc in docs_k]
        scores = cross_encoder.predict(pairs)
        best_docs.append(docs_k[int(np.argmax(scores))])

    return best_docs

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


# 4. Retrieve best docs for train and test

Running retrieval up front for all rows so it doesn't happen inside the training loop.

In [9]:
print("Retrieving best context documents for train set...")
train_best_docs = retrieve_best_doc(train['prompt'].tolist())
print(f"Done — {len(train_best_docs)} train docs retrieved")

print("Retrieving best context documents for test set...")
test_best_docs = retrieve_best_doc(test['prompt'].tolist())
print(f"Done — {len(test_best_docs)} test docs retrieved")

Retrieving best context documents for train set...
Done — 2000 train docs retrieved
Retrieving best context documents for test set...
Done — 500 test docs retrieved


# 5. RAG-augmented input formatting

Each of the 5 options is formatted as:
`Context: [best_doc] [SEP] [prompt] [SEP] [option_text]`

This gives the model three things in one input:
- the retrieved factual context
- the question
- the candidate answer
  
letting attention flow across all three.

In [10]:
def format_rag_choices(prompt, best_doc, row):
    """Build 5 RAG-augmented (context + prompt + option) strings for one row."""
    prefix = f"Context: {best_doc} [SEP] {prompt}"
    return [prefix + " [SEP] " + str(row[opt]) for opt in OPTION_COLS]

# Quick check on row 0
row0 = train.iloc[0]
sample_choices = format_rag_choices(row0['prompt'], train_best_docs[0], row0)
print("Sample formatted inputs for row 0:")
for opt, txt in zip(OPTION_COLS, sample_choices):
    print(f"  [{opt}] {txt[:120]}...")

Sample formatted inputs for row 0:
  [A] Context: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the...
  [B] Context: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the...
  [C] Context: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the...
  [D] Context: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the...
  [E] Context: Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the...


# 6. Tokenizer

In [11]:
MAX_LENGTH = 128  # balances context vs. sequence budget
tokenizer  = AutoTokenizer.from_pretrained("bert-base-uncased")
print(f"Tokenizer loaded - vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Tokenizer loaded — vocab size: 30522


# 7. Build Hugging Face Datasets for train + validation split

In [13]:
def make_records(df, best_docs, include_labels=True):
    """Tokenize all rows into {input_ids, attention_mask, labels} dicts."""
    records = []
    for i, (_, row) in enumerate(df.iterrows()):
        choices = format_rag_choices(str(row['prompt']), best_docs[i], row)
        enc = tokenizer(
            choices,
            padding    = "max_length",
            truncation = True,
            max_length = MAX_LENGTH,
        )
        record = {
            "input_ids"      : enc["input_ids"],       # list[5][MAX_LENGTH]
            "attention_mask" : enc["attention_mask"],
        }
        if include_labels:
            record["labels"] = int(LABEL_MAP[row['answer']])
        records.append(record)
    return records

In [14]:
# 80/20 train/val split for MAP@3 validation
from sklearn.model_selection import train_test_split

train_idx, val_idx = train_test_split(range(len(train)), test_size=0.2, random_state=42)

train_records = make_records(train.iloc[train_idx].reset_index(drop=True),
                              [train_best_docs[i] for i in train_idx])
val_records   = make_records(train.iloc[val_idx].reset_index(drop=True),
                              [train_best_docs[i] for i in val_idx])

hf_train = Dataset.from_list(train_records).with_format("torch")
hf_val   = Dataset.from_list(val_records).with_format("torch")

print(f"hf_train: {len(hf_train)} rows  hf_val: {len(hf_val)} rows")
print(f"First item input_ids shape: {len(hf_train[0]['input_ids'])} choices × {len(hf_train[0]['input_ids'][0])} tokens")

hf_train: 1600 rows  hf_val: 400 rows
First item input_ids shape: 5 choices × 128 tokens


# 8. Build LoRA model on top of AutoModelForMultipleChoice

In [15]:
base_model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

lora_config = LoraConfig(
    r              = 8,
    lora_alpha     = 16,
    target_modules = ["query", "value"],
    lora_dropout   = 0.1,
    bias           = "none",
    task_type      = TaskType.SEQ_CLS
)

lora_model = get_peft_model(base_model, lora_config)
lora_model.print_trainable_parameters()

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 295,681 || all params: 109,778,690 || trainable%: 0.2693


# 9. MAP@3 compute metric for Trainer

Plugged into the Trainer so validation MAP@3 is logged after every eval step.

In [16]:
from transformers import EvalPrediction

def compute_map3(eval_pred: EvalPrediction):
    logits, labels = eval_pred
    # logits shape: (n, 5)
    ranked_idx = np.argsort(-logits, axis=1)   # descending
    scores = []
    for label, ranks in zip(labels, ranked_idx):
        top3 = list(ranks[:3])
        scores.append(1 / (top3.index(label) + 1) if label in top3 else 0)
    return {"map3": round(float(np.mean(scores)), 4)}

# 10. Fine-tune with Hugging Face Trainer

In [17]:
import wandb
from kaggle_secrets import UserSecretsClient

# Pull the API key from Kaggle secrets
secrets = UserSecretsClient()
wandb_api_key = secrets.get_secret("WANDB_API_KEY")
wandb.login(key=wandb_api_key)

wandb.init(
    project = "smart-mcq-solver",
    name    = "rag-lora-bert-final",
    config  = {
        "model"           : "bert-base-uncased",
        "lora_r"          : 8,
        "lora_alpha"      : 16,
        "target_modules"  : ["query", "value"],
        "max_length"      : 128,
        "epochs"          : 3,
        "lr"              : 2e-4,
        "batch_size"      : 8,
        "grad_accum"      : 2,
        "retrieval_k"     : 5,
        "blend_threshold" : 0.05,
        "length_weight"   : 0.30,
    }
)
print(f"wandb run: {wandb.run.name}  —  {wandb.run.url}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: W&B API key is configured. Use `wandb login --relogin` to force relogin


wandb run: rag-lora-bert-final  —  https://wandb.ai/24f3003354-dl-genai-project/smart-mcq-solver/runs/fjefojmy


In [19]:
training_args = TrainingArguments(
    output_dir                  = "/kaggle/working/final_lora_output",
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 16,
    gradient_accumulation_steps = 2,       # effective batch = 16
    num_train_epochs            = 3,
    learning_rate               = 2e-4,
    warmup_ratio                = 0.1,
    weight_decay                = 0.01,
    eval_strategy               = "epoch",
    save_strategy               = "epoch",
    load_best_model_at_end      = True,
    metric_for_best_model       = "map3",
    greater_is_better           = True,
    logging_steps               = 20,
    report_to                   = "wandb",
    fp16                        = torch.cuda.is_available(),
)

trainer = Trainer(
    model            = lora_model,
    args             = training_args,
    train_dataset    = hf_train,
    eval_dataset     = hf_val,
    compute_metrics  = compute_map3,
)

print("Starting training...")
trainer.train()
print("Training complete.")

wandb.finish()

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Map3
1,5.945340,2.363125,0.739600
2,3.842729,1.876529,0.766700
3,3.945626,1.856650,0.765400


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]
/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training complete.


eval/loss,█▁▁
eval/map3,▁██
eval/runtime,▁▇█
eval/samples_per_second,█▂▁
eval/steps_per_second,█▂▁
train/epoch,▁▂▃▃▄▅▅▆▇██
train/global_step,▁▂▃▃▄▅▅▆▇██
train/grad_norm,▁▂▄█▄█▆
train/learning_rate,█▇▆▅▃▂▁
train/loss,█▇▄▂▁▁▂
eval/loss,1.85665


# 11. Validation MAP@3 check before touching test set

Same 'know before you submit' pattern from V4 — row-by-row correctness on held-out data.

In [21]:
lora_model.eval()

def predict_top3(df_subset, best_docs_subset):
    """Run inference and return (top3_letters_list, probs_list) for each row."""
    all_top3   = []
    all_probs  = []

    for i, (_, row) in enumerate(df_subset.iterrows()):
        choices = format_rag_choices(str(row['prompt']), best_docs_subset[i], row)
        enc = tokenizer(
            choices,
            padding    = "max_length",
            truncation = True,
            max_length = MAX_LENGTH,
            return_tensors = "pt"
        )
        input_ids   = enc["input_ids"].unsqueeze(0).to(DEVICE)
        attn_mask   = enc["attention_mask"].unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            logits = lora_model(input_ids=input_ids, attention_mask=attn_mask).logits[0]

        probs      = F.softmax(logits, dim=-1).cpu().numpy()
        ranked_idx = np.argsort(-probs)
        top3       = [IDX_TO_OPT[j] for j in ranked_idx[:3]]

        all_top3.append(top3)
        all_probs.append(probs)

    return all_top3, all_probs

In [22]:
val_df   = train.iloc[val_idx].reset_index(drop=True)
val_docs = [train_best_docs[i] for i in val_idx]

print("Running inference on validation set...")
val_top3, val_probs = predict_top3(val_df, val_docs)

# Row-by-row correctness table
check = val_df[['id', 'prompt', 'answer']].copy()
check['top3']           = [' '.join(t) for t in val_top3]
check['top1_correct']   = [t[0] == a for t, a in zip(val_top3, val_df['answer'])]
check['in_top3']        = [a in t  for t, a in zip(val_top3, val_df['answer'])]

n = len(check)
print(f"Top-1 exact match : {check['top1_correct'].sum()} / {n}  ({check['top1_correct'].mean():.3f})")
print(f"Correct in top-3  : {check['in_top3'].sum()} / {n}  ({check['in_top3'].mean():.3f})")
print(f"Wrong (not top-3) : {(~check['in_top3']).sum()} / {n}  ({(~check['in_top3']).mean():.3f})")

map3_scores = [
    1 / (list(t).index(a) + 1) if a in t else 0
    for t, a in zip(val_top3, val_df['answer'])
]
val_map3 = round(np.mean(map3_scores), 4)
print(f"\nValidation MAP@3: {val_map3}")

Running inference on validation set...
Top-1 exact match : 265 / 400  (0.662)
Correct in top-3  : 359 / 400  (0.897)
Wrong (not top-3) : 41 / 400  (0.102)

Validation MAP@3: 0.7667


# 12. Length tiebreaker blend

Content-word length scored 0.48 on its own and consistently outperformed cosine-based
signals. Used here as a lightweight tiebreaker: when the model's top-2 option probabilities
are very close (within a threshold), the longer option wins.

Rather than a fixed blend, this applies the length signal only where the model is uncertain —
preserving confident model predictions while rescuing ambiguous cases.

In [24]:
def content_word_len(text):
    text = str(text).lower()
    text = text.translate(str.maketrans('', '', string.punctuation))
    words = [w for w in text.split() if w not in ENGLISH_STOP_WORDS]
    return len(words)

def length_score_row(row):
    """Per-row min-max normalized content-word length for each option."""
    lengths = np.array([content_word_len(str(row[opt])) for opt in OPTION_COLS],
                       dtype=float)
    span = lengths.max() - lengths.min()
    if span == 0:
        return np.full(5, 0.5)
    return (lengths - lengths.min()) / span

BLEND_THRESHOLD = 0.05   # if top-2 model probs differ by less than this, blend in length
LENGTH_WEIGHT   = 0.30   # weight given to length when blending (model gets 0.70)

def blend_scores(model_probs, length_scores):
    """
    Blend model probabilities with length scores, but only when the model
    is uncertain (top-2 probs are within BLEND_THRESHOLD of each other).
    """
    sorted_probs = np.sort(model_probs)[::-1]
    if sorted_probs[0] - sorted_probs[1] < BLEND_THRESHOLD:
        # Model is uncertain — let length contribute
        return (1 - LENGTH_WEIGHT) * model_probs + LENGTH_WEIGHT * length_scores
    return model_probs  # Model is confident — trust it

print(f"Tiebreaker: blend length at {LENGTH_WEIGHT:.0%} when top-2 model probs differ < {BLEND_THRESHOLD}")

Tiebreaker: blend length at 30% when top-2 model probs differ < 0.05


# 13. Generate final predictions on test set

In [25]:
print("Running inference on test set with length tiebreaker...")

test_predictions = []

for i, (_, row) in enumerate(test.iterrows()):
    choices = format_rag_choices(str(row['prompt']), test_best_docs[i], row)
    enc = tokenizer(
        choices,
        padding    = "max_length",
        truncation = True,
        max_length = MAX_LENGTH,
        return_tensors = "pt"
    )
    input_ids  = enc["input_ids"].unsqueeze(0).to(DEVICE)
    attn_mask  = enc["attention_mask"].unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = lora_model(input_ids=input_ids, attention_mask=attn_mask).logits[0]

    model_probs  = F.softmax(logits, dim=-1).cpu().numpy()
    len_scores   = length_score_row(row)
    final_scores = blend_scores(model_probs, len_scores)

    ranked_idx = np.argsort(-final_scores)
    top3       = [IDX_TO_OPT[j] for j in ranked_idx[:3]]
    test_predictions.append(' '.join(top3))

    if i % 100 == 0:
        print(f"  {i}/{len(test)} done...")

print("Inference complete.")

Running inference on test set with length tiebreaker...
  0/500 done...
  100/500 done...
  200/500 done...
  300/500 done...
  400/500 done...
Inference complete.


# 14. Build and save submission.csv

In [27]:
submission = pd.DataFrame({
    'ID': test['id'],
    'Prediction': test_predictions
})

submission.to_csv('submission.csv', index=False)
submission.shape

(500, 2)